In [1]:
import wandb
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from nnModel import Net

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score


In [ ]:
run = wandb.init(
    entity="sanjana-learning",
    project="cat-or-dog",
    config={
        "architecture": "CNN",
        "dataset": "https://www.kaggle.com/datasets/shaunthesheep/microsoft-catsvsdogs-dataset/data",
    },
)

In [ ]:
# img_size = 50
# batch_size = 100 #images passing through at once 
# epochs = 2  #number of times entire dataset is passed through the model

In [ ]:
#overfitting do a validation split 
#check and see how many are being classified correctly and incorrectly
#save multiple models and compare them
#make graphs for each one

In [ ]:
# -------------------------
# 1) Load data
# -------------------------
X_data = np.load("model/catdog_X.npy", allow_pickle=True)
y_data = np.load("model/catdog_y.npy", allow_pickle=True)

# X should become [N, 1, 50, 50] float32
X = torch.tensor(np.array(X_data), dtype=torch.float32)
y = torch.tensor(np.array(y_data), dtype=torch.float32)  # [N,2]
y = y.argmax(dim=1).long()   

# If your y is shape [N,1], flatten to [N]
y = y.view(-1)

# If your X is currently [N, 50, 50], add channel dim -> [N,1,50,50]
if X.ndim == 3:
    X = X.unsqueeze(1)

print("X:", X.shape, X.dtype)
print("y:", y.shape, y.dtype, "unique:", torch.unique(y))
assert len(X) == len(y)


X: torch.Size([24946, 1, 50, 50]) torch.float32
y: torch.Size([24946]) torch.int64 unique: tensor([0, 1])


In [ ]:
# -------------------------
# 2) Split train/test
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y.numpy()  # keeps class balance similar in train/test
)

np.save("catdog_X_test.npy", X_test)
np.save("catdog_y_test.npy", y_test)

In [ ]:
# from sklearn.model_selection import RepeatedStratifiedKFold
# k = 5
# repeats = 2
# epochs = 2
# batch_size = 100
# lr = 1e-3
# seed = 42

# rskf = RepeatedStratifiedKFold(n_splits=k, n_repeats=repeats, random_state=seed)


In [5]:
from sklearn.model_selection import KFold 

kf = KFold(n_splits=10) 


In [6]:
# -------------------------
# 3) Setup model + training
# -------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
net = Net().to(device)

optimizer = optim.Adam(net.parameters(), lr=1e-3)
loss_function = nn.CrossEntropyLoss()

batch_size = 100
epochs = 5

In [14]:
kf = KFold(n_splits=5) 
for train_index, val_index in kf.split(X_train):
    X_train_fold, X_val_fold = X_train[train_index], X_train[val_index]
    y_train_fold, y_val_fold = y_train[train_index], y_train[val_index]
    
    # Now use X_train_fold, X_val_fold, y_train_fold, y_val_fold in your training loop


In [15]:
# -------------------------
# 5) Training loop
# -------------------------

#i am currently passing the entire dataset through the model each epoch 
# would it be better to do k-fold cross validation?
validation_scores = []
kf = KFold(n_splits=5) 
for train_index, val_index in kf.split(X_train):
    X_train_fold, X_val_fold = X_train[train_index], X_train[val_index]
    y_train_fold, y_val_fold = y_train[train_index], y_train[val_index]

    for epoch in range(epochs):
        net.train() # set train mode for things like dropout/batchnorm \

        # shuffle indices each epoch
        #this is important because it prevents the model from learning any order-based patterns in the data 
        # and helps improve generalization by exposing the model to different combinations of samples in each epoch.
        perm = torch.randperm(len(X_train_fold))
        X_train_shuf = X_train_fold[perm]
        y_train_shuf = y_train_fold[perm]

        #this is a common way to track the average loss across an epoch, which gives you a better sense of how the model
        # is learning over time, rather than just looking at the loss of the last batch.
        running_loss = 0.0

        #for each batch, we zero the gradients, compute the logits, calculate the loss, backpropagate, 
        # and update the weights.

        for i in range(0, len(X_train_shuf), batch_size):
            batch_X = X_train_shuf[i:i+batch_size].to(device) #is to(device) necessary?
            batch_y = y_train_shuf[i:i+batch_size].to(device)

        #the reason we call optimizer.zero_grad() before the backward pass is to reset the gradients
        # of all model parameters to zero.
        #optimizer.zero_grad(set_to_none=True) - Clears the gradients from the previous iteration. 
        # Without this, gradients accumulate and corrupt your updates.

        # logits = net(batch_X) - Passes the input batch through the network to get predictions 
        # (raw output before softmax).

        # loss = loss_function(logits, batch_y) - Computes how wrong the predictions are compared to the true labels.

        # loss.backward() - Performs backpropagation, computing gradients (∂loss/∂weight) 
        # for every parameter in the network.

        # optimizer.step() - Updates all the weights using the computed gradients. 
        # The optimizer adjusts weights to reduce the loss.
        # In sequence: Zero → Forward pass → Loss → Backward pass → Weight update. This repeats for each batch, gradually improving the model.
                    
            optimizer.zero_grad(set_to_none=True)
            logits = net(batch_X)                     # logits shape [B,2]
            loss = loss_function(logits, batch_y)     # batch_y shape [B]
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * batch_X.size(0)

        train_loss = running_loss / len(X_train_shuf)
        print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f}")

        # -------------------------
        # 6) Evaluate once per epoch
        # -------------------------
        net.eval() #switches to evaluation mode
        with torch.no_grad(): #dont compute gradients (saves memory)
            X_te = X_val_fold.to(device) #moves data to GPU/CPU
            logits_te = net(X_te)  # [N,2] #foward pass  
            #convert logits to probablities and extract prob class 1, 
            # moves to CPU and converts to Numpy for downstream sklearn 
            # metrics
            probs_pos = torch.softmax(logits_te, dim=1)[:, 1].cpu().numpy()
            preds = torch.argmax(logits_te, dim=1).cpu().numpy() #get class with higher probablity 

            y_true = y_val_fold.cpu().numpy() #convert true labels from tensor to numpy 

            cm = confusion_matrix(y_true, preds) #which predictions correct/incorrect
            
            acc = accuracy_score(y_true, preds)
            prec = precision_score(y_true, preds, zero_division=0)
            rec = recall_score(y_true, preds, zero_division=0)
            f1 = f1_score(y_true, preds, zero_division=0)
            

            #AUC can error if a fold/test set has only 1 class (rare here, but safe)
            try:
                auc = roc_auc_score(y_true, probs_pos)
            except ValueError:
                auc = float("nan")

        run.log({"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, 'auc': auc,
                    "train_loss": train_loss, "confusion_matrix": cm, 'step':epoch+1})
        validation_scores.append({"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, 'auc': auc,
                    "train_loss": train_loss, "confusion_matrix": cm, 'step':epoch+1})
        print(
            f"Epoch {epoch+1}/{epochs} | train_loss={train_loss:.4f} | "
            f"acc={acc:.3f} prec={prec:.3f} rec={rec:.3f} f1={f1:.3f} auc={auc:.3f}\n"
            f"Confusion matrix:\n{cm}\n"
        )

    # evidence of improvement:
    # train_loss should decrease across epochs (printed each epoch)
    # Validation metrics (accuracy, precision, recall, f1, auc) should improve


Epoch 1/5 - Train Loss: 0.6928
Epoch 1/5 | train_loss=0.6928 | acc=0.561 prec=0.558 rec=0.496 f1=0.526 auc=0.583
Confusion matrix:
[[740 448]
 [574 566]]

Epoch 2/5 - Train Loss: 0.6810
Epoch 2/5 | train_loss=0.6810 | acc=0.578 prec=0.548 rec=0.782 f1=0.645 auc=0.608
Confusion matrix:
[[453 735]
 [248 892]]

Epoch 3/5 - Train Loss: 0.6726
Epoch 3/5 | train_loss=0.6726 | acc=0.598 prec=0.578 rec=0.665 f1=0.619 auc=0.626
Confusion matrix:
[[635 553]
 [382 758]]

Epoch 4/5 - Train Loss: 0.6684
Epoch 4/5 | train_loss=0.6684 | acc=0.576 prec=0.548 rec=0.768 f1=0.639 auc=0.626
Confusion matrix:
[[466 722]
 [265 875]]

Epoch 5/5 - Train Loss: 0.6610
Epoch 5/5 | train_loss=0.6610 | acc=0.607 prec=0.573 rec=0.768 f1=0.657 auc=0.671
Confusion matrix:
[[536 652]
 [264 876]]

Epoch 1/5 - Train Loss: 0.6434
Epoch 1/5 | train_loss=0.6434 | acc=0.652 prec=0.629 rec=0.733 f1=0.677 auc=0.713
Confusion matrix:
[[671 500]
 [309 848]]

Epoch 2/5 - Train Loss: 0.6244
Epoch 2/5 | train_loss=0.6244 | acc=0.6

In [ ]:
validation_scores 

In [17]:
run.finish()

accuracy,▁▁▂▁▂▃▃▄▄▄▃▄▄▄▄▆▆▆▆▆█▇▇▇▇
auc,▁▁▂▂▃▃▄▄▄▄▅▅▅▅▅▇▆▆▆▆████▇
f1,▁▃▃▃▄▄▄▅▄▄▃▅▅▃▄▆▆▆▅▅█▇▇█▇
precision,▁▁▂▁▂▃▃▃▄▃▅▄▅▆▆▆▆▅▇▇██▇▆█
recall,▁▆▄▆▆▅▅▆▄▅▂▇▄▂▂▆▅▆▄▄█▆▇█▅
step,▁▃▅▆█▁▃▅▆█▁▃▅▆█▁▃▅▆█▁▃▅▆█
train_loss,████▇▇▇▇▆▆▆▆▅▅▅▅▄▄▃▃▃▃▂▁▁
accuracy,0.81822
auc,0.91585
f1,0.80799
precision,0.87426


In [ ]:

torch.save(net.state_dict(), "model/catdog_cnn_model_1.pth")

In [ ]:
#TODO 
#save model after each epoch/K
#assess on validation, train and test 
#updated the logs 

In [ ]:
import pickle

file_path = 'model/validationScores.pkl'

# Open the file in write-binary mode
with open(file_path, 'wb') as file:
    # Serialize and write the list to the file
    pickle.dump(validation_scores, file)

print(f"List successfully pickled to {file_path}")


In [ ]:
#import pickle
# # Open the file in read-binary mode
# with open(file_path, 'rb') as file:
#     # Read the file and deserialize the data back into a list
#     loaded_list = pickle.load(file)

In [ ]:
net = Net()
net.load_state_dict(torch.load("model/catdog_cnn_model_1.pth"))
net.eval()  # Set the model to evaluation mode

X_test_data = np.load("model/catdog_X_test.npy", allow_pickle=True)
y_test_data = np.load("model/catdog_y_test.npy", allow_pickle=True)

X_test = torch.Tensor([img for img in X_test_data])
y_test = torch.Tensor([dx for dx in y_test_data])

In [ ]:
net.eval() #switches to evaluation mode
with torch.no_grad(): #dont compute gradients (saves memory)
    X_te = X_test.to(device) #moves data to GPU/CPU
    logits_te = net(X_te)  # [N,2] #foward pass  
    #convert logits to probablities and extract prob class 1, 
    # moves to CPU and converts to Numpy for downstream sklearn 
    # metrics
    probs_pos = torch.softmax(logits_te, dim=1)[:, 1].cpu().numpy()
    preds = torch.argmax(logits_te, dim=1).cpu().numpy() #get class with higher probablity 

    y_true = y_val_fold.cpu().numpy() #convert true labels from tensor to numpy 

    cm = confusion_matrix(y_true, preds) #which predictions correct/incorrect
    
    acc = accuracy_score(y_true, preds)
    prec = precision_score(y_true, preds, zero_division=0)
    rec = recall_score(y_true, preds, zero_division=0)
    f1 = f1_score(y_true, preds, zero_division=0)
    